In [1]:
import pandas as pd
import os
import re
from openai import OpenAI
from anthropic import Anthropic
from together import Together
from multiprocessing.pool import ThreadPool
import ujson as json
from tqdm import tqdm


In [2]:
train = pd.read_json(path_or_buf='test.jsonl', lines=True)
sample = train.sample(n=100, random_state=1)

client = OpenAI(
  api_key=os.environ.get("TOGETHER_API_KEY"),
  base_url="https://api.together.xyz/v1",
)

def get_response(msg, mod="mistralai/Mixtral-8x22B-Instruct-v0.1", temp=0):
    response = client.chat.completions.create(
      model=mod,
      messages=msg,
      temperature=temp
    )
    return response.choices[0].message.content

In [87]:
def self_consistency_solver(question, options, n_experts):
    instructions = f'''
    Imagine {n_experts} completely independent experts who reason differently
    are answering a question. The question and options are delimited by triple backticks.
    The final answer is obtained by majority vote.
    Step 1. For each of the experts, give their step-by-step
    reasoning and answer, choosing from the given options
    Step 2. Determine the final answer by majority vote
    Step 3. Return the final answer, obtained by majority vote,
    prefixed by 'Final answer:' and followed by the letter of the chosen option (A, B, C, D, or E). 
    IMPORTANT: only respond with 'Final answer: ' followed by the letter and no additional characters.
    '''
    user_content = f'```Question: {question}\nOptions: {options}```'
    msg = [
        {"role": "system", "content": instructions},
        {"role": "user", "content": user_content}
    ]
    return get_response(msg=msg, temp=0.5)


def identify_final_answer(question, options, solution):
    instructions = '''
    You will be provided with the answer to a question.
    The question and options are delimited by triple backticks,
    and the answer is delimited by triple hashtags.
    Extract the final answer from the provided solution.
    Return only the letter corresponding to the chosen option (A, B, C, D, or E),
    prefixed by 'Final answer:'
    IMPORTANT: only respond with 'Final answer: ' followed by the letter and no additional characters.
    '''
    try:
        # First, try to split by 'Final answer:'
        if 'Final answer:' in solution:
            answer = solution.split('Final answer:')[1].strip()
        else:
            # If 'Final answer:' is not found, use the entire solution
            answer = solution.strip()
        
        # Remove any leading/trailing whitespace and punctuation
        answer = re.sub(r'^[\s\W]+|[\s\W]+$', '', answer)
        
        # If the answer is just a single letter, use it directly
        if len(answer) == 1 and answer.upper() in 'ABCDE':
            return f'Final answer: {answer.upper()}'
        
        # Otherwise, wrap the answer in triple hashtags
        answer = f'###{answer}###'
        
        user_content = f'```Question: {question}\nOptions: {options}```{answer}'
        msg = [
            {"role": "system", "content": instructions},
            {"role": "user", "content": user_content}
        ]
        return get_response(msg=msg, temp=0)
    except Exception as e:
        print(f"Error in identify_final_answer: {e}")
        return 'NA'

In [88]:
def clean_answer(answer):
    # Remove everything but numbers in integer or decimal form
    answer_clean = re.sub('[^\d\.]', '', answer)
    # If the last character is a decimal, remove it, it was probably presented as a sentence
    if answer_clean[-1] == '.':
        answer_clean = answer_clean[:-1]
    # If number contains decimal, decide if it should be removed
    if '.' in answer_clean:
        # If the number contains only trailing zeroes, strip them and remove it
        if answer_clean[-1] == '0' and answer_clean[-2] == '0':
            answer_clean = answer_clean.rstrip('0')
            answer_clean = answer_clean[:-1]
        # If the decimal is now the last character, remove it
        if answer_clean[-1] == '.':
            answer_clean = answer_clean[:-1]
    return answer_clean


def parse_final_answer(evaluation):
    try:
        # Look for "Final answer:" or "Answer:" followed by a letter (with optional closing parenthesis)
        match = re.search(r'(?:Final answer:|Answer:)\s*([A-E])\)?', evaluation, re.IGNORECASE)
        if match:
            return match.group(1).upper()
        
        # Look for a letter followed by a closing parenthesis
        match = re.search(r'([A-E])\)', evaluation, re.IGNORECASE)
        if match:
            return match.group(1).upper()
        
        # Look for "Option X" or "X)" where X is A, B, C, D, or E
        match = re.search(r'(?:Option\s*|^)([A-E])(?:\)|:|\s|$)', evaluation, re.IGNORECASE)
        if match:
            return match.group(1).upper()
        
        # If no match is found, look for any standalone letter A-E
        options = re.findall(r'\b([A-E])\b', evaluation, re.IGNORECASE)
        if options:
            return options[-1].upper()  # Return the last mentioned option
        
        # If still no match, return 'NA'
        return 'NA'
    except Exception as e:
        print(f"Error in parse_final_answer: {e}")
        return 'NA'

def get_best_answer(options):
    answer_count = [[x, options.count(x)] for x in set(options) if x not in ['', 'NA']]
    answer_count_sorted = sorted(answer_count, key=lambda x: x[1], reverse=True)
    if len(answer_count_sorted) > 0:
        return answer_count_sorted[0][0]
    else:
        return 'NA'


def get_true_answer(answer):
    answer = answer.split('### ')[1]
    answer = re.sub('[^\d\.]', '', answer)
    return answer


def multi_step_solver(question, options, n_experts, n_attempts):
    # Pool for parallelization
    pool = ThreadPool(n_attempts)
    
    # Generate attempts
    attempts = pool.starmap(self_consistency_solver, zip([question]*n_attempts, [options]*n_attempts, [n_experts]*n_attempts))
    
    # Identify the final answers
    answers = pool.starmap(identify_final_answer, zip([question]*n_attempts, [options]*n_attempts, attempts))
    
    # Parse the final answers from both attempts and answers
    answers_parsed_attempts = [parse_final_answer(attempt) for attempt in attempts]
    answers_parsed_answers = [parse_final_answer(answer) for answer in answers]
    
    # Combine parsed answers, preferring non-'NA' results
    answers_parsed = [a if a != 'NA' else b for a, b in zip(answers_parsed_attempts, answers_parsed_answers)]
    
    # Identify the best answer
    best_answer = get_best_answer(answers_parsed)
    
    # Results
    return {
        "best_answer": best_answer,
        "attempts": attempts,
        "answers": answers,
        "answers_parsed": answers_parsed
    }

In [94]:
try:
    df = pd.read_json('answers.ndjson', lines=True)
    start = len(df[(df['n_experts'] == 1) & (df['n_attempts'] == 1)])
except:
    start = 0

for i in tqdm(range(start, len(sample))):
    for n_attempts in [1, 3, 5, 10]:
        for n_experts in [1, 3, 5]:
            question = sample.iloc[i].question
            options = sample.iloc[i].options
            true_answer = sample.iloc[i].correct
            results = multi_step_solver(question, options, n_experts=n_experts, n_attempts=n_attempts)
            with open(f'answers.ndjson', 'a+') as f:
                json.dump({
                    "n_experts": n_experts,
                    "n_attempts": n_attempts,
                    "question": question,
                    "options": options,
                    "true_answer": true_answer,
                    "best_answer": results['best_answer'],
                    "attempts": results['attempts'],
                    "answers": results['answers'],
                    "answers_parsed": results['answers_parsed']
                }, f)
                f.write('\n')

  0%|          | 0/94 [00:00<?, ?it/s]

 12%|█▏        | 11/94 [32:49<4:07:38, 179.01s/it]


KeyboardInterrupt: 

In [14]:
# Read the NDJSON file
with open('/p/llmreliability/test_repos/llmpromptboosting/SoK_Self_Consistency/scripts/answers.ndjson', 'r') as file:
    df = pd.DataFrame([json.loads(line) for line in file])

# Perform the transformations
df['is_correct'] = df['best_answer'] == df['true_answer']
df['n_experts'] = df['n_experts'].astype('category')
df['n_attempts'] = df['n_attempts'].astype('category')

In [15]:
len(df)

23

In [16]:
display(df)

,n_experts,n_attempts,question,options,true_answer,best_answer,attempts,answers,answers_parsed,is_correct
0,1,1,Assistants are needed to prepare for preparati...,"[A)8, B)10, C)12, D)15, E)19]",B,E,[**Expert 1:**\nStep 1: Determine the number o...,[Final answer: E],[E],False
1,3,1,Assistants are needed to prepare for preparati...,"[A)8, B)10, C)12, D)15, E)19]",B,A,[**Expert 1: Logical Reasoning**\n\nStep 1: De...,[Final answer: A],[A],False
2,5,1,Assistants are needed to prepare for preparati...,"[A)8, B)10, C)12, D)15, E)19]",B,E,[**Expert 1: Engineer**\nStep 1: Calculate the...,[Final answer: E],[E],False
3,1,3,Assistants are needed to prepare for preparati...,"[A)8, B)10, C)12, D)15, E)19]",B,D,[### Expert 1's Reasoning\n\n#### Step 1: Dete...,"[Final answer: D, Final answer: C, Final answe...","[D, C, D]",False
4,3,3,Assistants are needed to prepare for preparati...,"[A)8, B)10, C)12, D)15, E)19]",B,B,[**Expert 1: Mathematical Analyst**\n\nStep 1:...,"[Final answer: E, Final answer: B, Final answe...","[E, B, B]",True
5,5,3,Assistants are needed to prepare for preparati...,"[A)8, B)10, C)12, D)15, E)19]",B,B,[**Expert 1: Logical Thinker**\n\nStep 1: Let'...,"[Final answer: B, Final answer: B, Final answe...","[B, B, B]",True
6,1,5,Assistants are needed to prepare for preparati...,"[A)8, B)10, C)12, D)15, E)19]",B,A,[**Expert 1:**\n\nStep 1: Let's denote the num...,"[Final answer: A, Final answer: D), Final answ...","[A, D, A, B, D]",False
7,3,5,Assistants are needed to prepare for preparati...,"[A)8, B)10, C)12, D)15, E)19]",B,B,[**Expert 1: Logical Reasoner**\n\nStep 1: Det...,"[Final answer: B), Final answer: B, Final answ...","[B, B, B, D, E]",True
8,5,5,Assistants are needed to prepare for preparati...,"[A)8, B)10, C)12, D)15, E)19]",B,B,[Here are the step-by-step reasoning and answe...,"[Final answer: B, Final answer: B, Final answe...","[B, B, B, B, B]",True
9,1,10,Assistants are needed to prepare for preparati...,"[A)8, B)10, C)12, D)15, E)19]",B,A,[**Expert 1's Reasoning**\n\nTo determine the ...,"[Final answer: C, Final answer: A, Final answe...","[C, A, E, A, D, C, D, C, B, A]",False


In [17]:
df.groupby(['n_experts', 'n_attempts']).size().reset_index(name='n').query('n == 100')

,n_experts,n_attempts,n


In [7]:
# Filter the DataFrame
filtered_df = df[(df['n_experts'] == 1) & (df['n_attempts'] == 1)]

# Calculate the percentage of correct answers
if len(filtered_df) > 0:
    pct_correct = filtered_df['is_correct'].mean() * 100
    print(f"Percentage correct: {pct_correct:.2f}%")
else:
    print("No rows match the condition")

# Additional checks
print(f"Total rows in filtered DataFrame: {len(filtered_df)}")
print(f"Value counts of 'is_correct' in filtered DataFrame:")
print(filtered_df['is_correct'].value_counts(normalize=True) * 100)

Percentage correct: 58.67%
Total rows in filtered DataFrame: 150
Value counts of 'is_correct' in filtered DataFrame:
is_correct
True     58.666667
False    41.333333
Name: proportion, dtype: float64


In [ ]:
import pandas as pd
import json

# Read the NDJSON file
with open('/p/llmreliability/test_repos/llmpromptboosting/SoK_Self_Consistency/scripts/claude_answers.ndjson', 'r') as file:
    df = pd.DataFrame([json.loads(line) for line in file])

# Perform the initial transformations
df['is_correct'] = df['best_answer'] == df['true_answer']
df['n_experts'] = df['n_experts'].astype('category')
df['n_attempts'] = df['n_attempts'].astype('category')

# Print the total number of rows
print(f"Total number of rows: {len(df)}")

# Analysis for n_experts = 1 and n_attempts = 1
filtered_df = df[(df['n_experts'] == 1) & (df['n_attempts'] == 1)]
if len(filtered_df) > 0:
    pct_correct = filtered_df['is_correct'].mean() * 100
    print(f"\nPercentage correct for n_experts=1 and n_attempts=1: {pct_correct:.2f}%")
else:
    print("\nNo rows match the condition n_experts=1 and n_attempts=1")

print(f"Total rows in filtered DataFrame: {len(filtered_df)}")
print("Value counts of 'is_correct' in filtered DataFrame:")
print(filtered_df['is_correct'].value_counts(normalize=True) * 100)

# Group by n_attempts and calculate percentage correct
summary = df.groupby('n_attempts')['is_correct'].mean().reset_index()
summary['pct_correct'] = summary['is_correct'] * 100

# Format the percentage to two decimal places
summary['pct_correct'] = summary['pct_correct'].round(2)

# Rename columns for clarity
summary = summary.rename(columns={'n_attempts': '# of Attempts', 'pct_correct': '% Correct Answers'})

# Drop the 'is_correct' column as it's no longer needed
summary = summary.drop('is_correct', axis=1)

# Display the summary
print("\nSummary of correct answers by number of attempts:")
print(summary.to_string(index=False))

Total number of rows: 1385

Percentage correct for n_experts=1 and n_attempts=1: 59.48%
Total rows in filtered DataFrame: 116
Value counts of 'is_correct' in filtered DataFrame:
is_correct
True     59.482759
False    40.517241
Name: proportion, dtype: float64

Summary of correct answers by number of attempts:
# of Attempts  % Correct Answers
            1              58.05
            3              65.23
            5              71.30
           10              70.93
